# Viv in a notebook: `%%viv` and run records

`%load_ext vipyr` adds the `%%viv NAME` cell magic. A cell of Viv defs becomes the module `NAME`;
its functions take NumPy arrays and pandas or polars DataFrames (columns bind to parameters by name).
Each call appends a run record to `.viv_runs.jsonl` beside this notebook. A second call with the same
inputs that gives a different output prints a warning naming both records.

Needs the `viv` compiler (`$VIV`, `viv` on PATH, or a checkout's `target/release/viv`) and a C compiler.


In [ ]:
import numpy as np
import pandas as pd

%load_ext vipyr

A feature per transaction: the user's mean amount over their *earlier* transactions,
and how many there were. The rows are sorted by user, then time.

In [ ]:
%%viv feats
record Prior {
    mean_prev: Arr
    n_prev: [Int]
}

#| Mean amount of each user's earlier rows (0 when there are none).
def prior_mean(user: [Int], amount: Arr) -> Prior {
    n = len(amount)
    mean_prev = zeros(n)
    n_prev = int_zeros(n)
    total = 0.0
    count: Int = 0
    for i to n {
        if i > 0 and user[i] != user[i - 1] {
            total = 0.0
            count = 0
        }
        if count > 0 {
            mean_prev[i] = total / count
        }
        n_prev[i] = count
        total += amount[i]
        count += 1
    }
    return Prior { mean_prev: mean_prev, n_prev: n_prev }
}

In [ ]:
df = pd.DataFrame({
    "user":   [1, 1, 1, 2, 2, 3],
    "amount": [10.0, 30.0, 20.0, 5.0, 7.0, 99.0],
})
out = feats.prior_mean(df)
df.join(out)

Each call left a record: the function, the SHA-256 of the cell, the compiler, a hash per input column,
the output hash and the time.

In [ ]:
import json
runs = [json.loads(line) for line in open(".viv_runs.jsonl")]
last = runs[-1]
{k: last[k] for k in ("id", "program", "inputs_hash", "outputs_hash", "time")}, last["inputs"]["args"][0]["columns"]

Calling again on the same frame gives the same output hash, so the record has the same `id` and there is
no warning. A function that read something besides its arguments would print `rerun mismatch` here.

In [ ]:
feats.prior_mean(df)
runs = [json.loads(line) for line in open(".viv_runs.jsonl")]
runs[-1]["id"] == runs[-2]["id"], "mismatch_of" in runs[-1]

The cell as one C file, the source of the shared library the notebook calls:

In [ ]:
%viv_export feats -o prior_mean.c

`%viv_runs PATH` sends the records elsewhere; `%viv_runs off` stops recording.

In [ ]:
%viv_runs